### **Bài tập 3**

Viết code trên python để phân tích và vẽ cây cú pháp phụ thuộc cho câu sau:

1. **"Hà Nội, thủ đô của Việt Nam, luôn thu hút nhiều du khách"**
2. **"Con cáo nhanh nhẹn, vốn đang núp sau bụi rậm, bỗng nhảy qua con chó lười và làm các chú chim trong vườn sợ hãi."**

Hãy sử dụng thư viện: **Stanza** và **VNcoreNLP**

Yêu cầu: sau khi thực hiện xong chương trình, lập báo cáo gồm các phần:

- **Phần 1: Stanza** (giới thiệu thư viện, code chương trình, chụp kết quả màn hình)
- **Phần 2: VNcoreNLP** (giới thiệu thư viện, code chương trình, chụp kết quả màn hình)


In [ ]:
# # Install Stanza
# !pip install stanza

# # Install VNcoreNLP and Java
# !pip install vncorenlp
# !apt-get install openjdk-11-jdk-headless -qq > /dev/null

In [ ]:
# !pip install cairosvg

#### Phải có Java 8+ thôi, vì bản 17, 21, 24 nó không ăn thư viện được và tạp khá lâu.

In [ ]:
import os
from vncorenlp import VnCoreNLP
import pandas as pd
from spacy.tokens import Doc
from spacy.vocab import Vocab
from spacy import displacy
# import cairosvg
# from google.colab import files

vncorenlp_dir = 'D:/PythonIDLE/VnCoreNLP-master'
# if not os.path.exists(vncorenlp_dir):
#     print("Downloading VnCoreNLP models...")
#     !git clone https://github.com/vncorenlp/VnCoreNLP.git /content/VnCoreNLP

if not os.path.exists(vncorenlp_dir):
    print("Lỗi ko tồn tại thu mục %s"%(vncorenlp_dir))
vncorenlp_path = os.path.join(vncorenlp_dir, 'VnCoreNLP-1.2.jar')
parser = VnCoreNLP(vncorenlp_path, annotators="wseg,pos,ner,parse", max_heap_size = '-Xmx500m')

sentence1 = "Hà Nội, thủ đô của Việt Nam, luôn thu hút nhiều du khách."
sentence2 = "Con cáo nhanh nhẹn, vốn đang núp sau bụi rậm, bỗng nhảy qua con chó lười và làm các chú chim trong vườn sợ hãi."

def process_sentence_vncorenlp(sentence, parser):
    print("Đang tạo Parser VNCoreNLP"); parsed = parser.annotate(sentence); print("Đã xong")
    tokens = parsed['sentences'][0] ; rows = []
    for word in tokens:
        idx = int(word["head"]) - 1
        head = tokens[idx]["form"] if idx >= 0 else "ROOT"
        rows.append({
            "TEXT": word["form"],
            "LEMMA": word["form"].lower(),
            "POS": word["posTag"],
            "HEAD": head,
            "DEPREL": word["depLabel"]
        })

    df = pd.DataFrame(rows)
    print(df)
    vocab = Vocab()
    words = [t["form"] for t in tokens]
    spaces = [True] * len(words)
    heads = [int(t["head"]) - (i+1) for i, t in enumerate(tokens)]
    deps = [t["depLabel"] for t in tokens]

    doc = Doc(vocab, words=words, spaces=spaces)
    for i, token in enumerate(doc):
        token.dep_ = deps[i]
        head_idx = i + heads[i]
        if head_idx >= 0 and head_idx < len(doc) and head_idx != i:
             token.head = doc[head_idx]
        else:
            token.head = token
    displacy.render(doc, style="dep", jupyter=True)

    # Export to SVG
    # Use jupyter=False and page=True to get the full SVG string
    svg = displacy.render(doc, style="dep", jupyter=False)

    return svg, df

# Process the first sentence
print("--- VNcoreNLP - Sentence 1 ---")
svg1, df1 = process_sentence_vncorenlp(sentence1, parser)
svg_path1 = "/content/dep_tree_sentence1.svg"
png_path1 = "/content/dep_tree_sentence1.png"
with open(svg_path1, "w", encoding="utf-8") as f:
    f.write(svg1)

# Process the second sentence
print("\n--- VNcoreNLP - Sentence 2 ---")
svg2, df2 = process_sentence_vncorenlp(sentence2, parser)
svg_path2 = "/content/dep_tree_sentence2.svg"
png_path2 = "/content/dep_tree_sentence2.png"
with open(svg_path2, "w", encoding="utf-8") as f:
    f.write(svg2)

parser.close()

Đang tạo Parser


KeyboardInterrupt: 